In [1]:
# Import required libraries
import sys
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime

# Import the LLM debiasing analyzer
from LLM_debias import LLMPositionBiasAnalyzer

print("📚 Libraries imported successfully!")
print(f"📅 Experiment started at: {datetime.now()}")


📚 Libraries imported successfully!
📅 Experiment started at: 2025-07-26 14:57:09.236882


In [2]:
# # Data preprocessing: Load books data and create user-title-timestamp format
# print("🔧 Processing Books dataset...")

# # Step 1: Read the ratings CSV (columns: user_id, item_id, rating, timestamp)
# ratings_data_file = 'data/books/ratings_Books.csv'
# ratings_df = pd.read_csv(ratings_data_file, header=None, names=['UserID', 'item', 'rating', 'Timestamp'])

# print(f"📊 Loaded {len(ratings_df):,} ratings")
# print(f"📈 Unique users: {ratings_df['UserID'].nunique():,}")
# print(f"📚 Unique items: {ratings_df['item'].nunique():,}")

# # Step 2: Read the JSONL metadata
# metadata_json_file = 'data/books/meta_Books.jsonl'
# item_title_map = {}

# print("📖 Loading book metadata...")
# with open(metadata_json_file, 'r') as f:
#     for line in f:
#         try:
#             data = json.loads(line)
#             if 'parent_asin' in data and 'title' in data:
#                 asin = data['parent_asin'].strip()
#                 item_title_map[asin] = data['title']
#         except json.JSONDecodeError:
#             continue

# print(f"📚 Loaded metadata for {len(item_title_map):,} books")

# # Step 3: Map ASIN -> Title
# ratings_df['Title'] = ratings_df['item'].astype(str).map(item_title_map)

# # Step 4: Filter only those rows that matched
# filtered_df = ratings_df.dropna(subset=['Title'])
# print(f"📊 Matched {len(filtered_df):,} ratings with book titles ({len(filtered_df)/len(ratings_df)*100:.1f}%)")

# # Step 5: Final dataset
# final_df = filtered_df[['UserID', 'Title', 'Timestamp']]

# # Save the processed data
# output_path = 'data/books/user_title_timestamp.csv'
# final_df.to_csv(output_path, index=False)
# print(f"💾 Saved processed data to {output_path}")

# print("\n📊 Sample of processed data:")
# print(final_df.head())


In [3]:
final_df = pd.read_csv('data/books/user_title_timestamp.csv')

In [4]:
# Initialize the LLM Position Bias Analyzer for books
print("🚀 Initializing LLM Position Bias Analyzer for Books...")

analyzer = LLMPositionBiasAnalyzer(
    data=final_df,
    data_name="books",
    model="gpt-3.5-turbo",
    backend="openai",
    list_size=20,
    api_tier="tier_1"
)

print("✅ Analyzer initialized successfully!")

🚀 Initializing LLM Position Bias Analyzer for Books...
📊 User filtering results:
  Total users in dataset: 7087342
  Users with ≥6 items: 479984
  Filtered out: 6607358 users
✅ Selected 5 bias users and 200 evaluation users
   All selected users have ≥6 items for reliable evaluation
Initialized LLM Bias Analyzer:
  Model: gpt-3.5-turbo
  Backend: openai
  API Tier: tier_1
  Rate Limits: 3500 RPM, 1000000 TPM
  Max Workers: 8
  Batch Size: 15
  Request Delay: 0.200s
✅ Analyzer initialized successfully!


In [5]:
# bias_analysis =  analyzer.compute_bias_analysis(5,None,True,None,20)
# print(bias_analysis)

In [6]:
prebias_gpt35_book = {'avg_primacy': 0.220,
 'avg_recency': 0.240,
 'avg_middle': 1.540}

In [9]:
# Main debiasing experiment with list size 20
print("\n🔧 COMPLETE DEBIASING EXPERIMENT - BOOKS")
print("=" * 50)

num_candidates = 20    # Number of candidates per evaluation
num_trials = 15       # Number of randomization trials per user
batch_size = 20        # Batch size for processing

print(f"🎯 Candidates per evaluation: {num_candidates}")
print(f"🔄 Trials per user: {num_trials}")
print(f"📦 Batch size: {batch_size}")

# Run the complete evaluation pipeline
print("\n🚀 RUNNING COMPLETE EVALUATION PIPELINE...")
print("This will:")
print("1. 📊 Select separate users for bias detection vs evaluation")
print("2. 🔍 Run bias detection on bias detection users")  
print("3. ⚖️ Calculate propensity scores from detected bias")
print("4. 📈 Evaluate on evaluation users using calculated propensity scores")
print("5. 💾 Save all raw data for future reanalysis")

# Uncomment and run the evaluation below
results = analyzer.evaluate_our_method_batched(
    num_candidates=num_candidates,
    num_trials=num_trials,
    aggregation_method="mean",
    batch_size=batch_size,
    use_parallel=True,
    precalculated_bias=prebias_gpt35_book,
    checkpoint_file="evaluation_checkpoint_books_trial15.json"
)

print("\n⚠️  Uncomment the evaluation code above to run the experiment")
print("✅ Setup complete - ready to run evaluation!")



🔧 COMPLETE DEBIASING EXPERIMENT - BOOKS
🎯 Candidates per evaluation: 20
🔄 Trials per user: 15
📦 Batch size: 20

🚀 RUNNING COMPLETE EVALUATION PIPELINE...
This will:
1. 📊 Select separate users for bias detection vs evaluation
2. 🔍 Run bias detection on bias detection users
3. ⚖️ Calculate propensity scores from detected bias
4. 📈 Evaluate on evaluation users using calculated propensity scores
5. 💾 Save all raw data for future reanalysis
📁 Checkpoint file: evaluation_checkpoint_books_trial15.json
API Tier: tier_1 (RPM: 3500, TPM: 1000000)
Max workers - Bias: 8, Trials: 4, Users: 1
🔄 Recalculating bias analysis with new precalculated bias scores...
   Previous bias: {}
   New bias: {'avg_primacy': 0.22, 'avg_recency': 0.24, 'avg_middle': 1.54}
Bias users: ['AN9ZHVSV9KM4I', 'A28WK1N0HTLV2', 'AD7M7RRDWLBPH', 'A34QVU3D6FLYSR', 'A1U8A9X3SW9R6F']
Using precalculated bias scores...
👥 Total users: 200, Completed: 0, Remaining: 200

🔄 Processing batch 1/10 (20 users)
Evaluating 20 users in paral

Trials (batch 1/1): 100%|███████████████████████| 15/15 [00:14<00:00,  1.01it/s]


Completed 15 successful trials out of 15 attempted

Running 15 randomization trials (with raw data preservation)...
Executing 15 trials in parallel with max_workers=4...
Rate limiting: 4 workers, 0.200s delay, batch size: 15


Trials (batch 1/1): 100%|███████████████████████| 15/15 [00:11<00:00,  1.33it/s]


Completed 15 successful trials out of 15 attempted

Running 15 randomization trials (with raw data preservation)...
Executing 15 trials in parallel with max_workers=4...
Rate limiting: 4 workers, 0.200s delay, batch size: 15


...

Completed 15 successful trials out of 15 attempted

Running 15 randomization trials (with raw data preservation)...
Executing 15 trials in parallel with max_workers=4...
Rate limiting: 4 workers, 0.200s delay, batch size: 15


Trials (batch 1/1): 100%|███████████████████████| 15/15 [00:11<00:00,  1.27it/s]


Completed 15 successful trials out of 15 attempted

Running 15 randomization trials (with raw data preservation)...
Executing 15 trials in parallel with max_workers=4...
Rate limiting: 4 workers, 0.200s delay, batch size: 15


Trials (batch 1/1): 100%|███████████████████████| 15/15 [00:11<00:00,  1.27it/s]


Completed 15 successful trials out of 15 attempted
Progress: 20/20 users completed
✅ Completed evaluation of 20 users
✅ Batch 10 completed. Progress: 200/200 users

📊 Computing final metrics from 200 user results...

OUR METHOD EVALUATION RESULTS vs BENCHMARKS

Our Method Results:
  Accuracy:    0.1850 ± 0.3883
  NDCG@1:      0.1850 ± 0.3883
  NDCG@5:      0.3027 ± 0.3944
  NDCG@10:     0.3979 ± 0.3453
  NDCG@20:     0.4705 ± 0.2734
  Number of evaluations: 200

Benchmark Results (Accuracy) - From Paper:
Method          Movie Dataset  
------------------------------
Raw Output      0.2740±0.0593
Bootstrapping   0.2537
STELLA          0.2976
Our Method      0.1850±0.3883

Accuracy Comparison (Movie Dataset):
----------------------------------------
Our Method vs Raw Output:    -0.0890
Our Method vs Bootstrapping: -0.0687
Our Method vs STELLA:        -0.1126

NDCG Analysis:
----------------------------------------
NDCG@1 = Accuracy: 0.1850
NDCG@5:  0.3027 (163.6% of NDCG@1)
NDCG@10: 0.39

In [ ]:
# Compute bias analysis with 100 bias users (list size 20)
print("🔍 Computing bias analysis with 100 bias users...")

# Uncomment the line below to run extended bias analysis
# bias_analysis_100 = analyzer.compute_bias_analysis(100, None, True, None, 20)
# print(bias_analysis_100)

# Placeholder values for 100 user bias analysis - replace with actual results
prebias_gpt35_books_100 = {
    'avg_primacy': 0.148,  # Placeholder - replace with actual computed values
    'avg_recency': 0.160,  # Placeholder - replace with actual computed values
    'avg_middle': 1.692    # Placeholder - replace with actual computed values
}

print("📊 Extended bias scores (100 users - placeholder):")
print(f"  Primacy: {prebias_gpt35_books_100['avg_primacy']:.3f}")
print(f"  Recency: {prebias_gpt35_books_100['avg_recency']:.3f}")
print(f"  Middle: {prebias_gpt35_books_100['avg_middle']:.3f}")


In [ ]:
# Main evaluation with extended bias analysis (100 users)
print("\n🔧 COMPLETE DEBIASING EXPERIMENT - BOOKS (Extended)")
print("=" * 60)

num_candidates = 20    # Number of candidates per evaluation
num_trials = 20        # Number of randomization trials per user
batch_size = 20        # Batch size for processing

print(f"🎯 Candidates per evaluation: {num_candidates}")
print(f"🔄 Trials per user: {num_trials}")
print(f"📦 Batch size: {batch_size}")

# Run the complete evaluation pipeline with extended bias
print("\n🚀 RUNNING COMPLETE EVALUATION PIPELINE (Extended)...")
print("This will:")
print("1. 📊 Select separate users for bias detection vs evaluation")
print("2. 🔍 Run bias detection on bias detection users (100 users)")  
print("3. ⚖️ Calculate propensity scores from detected bias")
print("4. 📈 Evaluate on evaluation users using calculated propensity scores")
print("5. 💾 Save all raw data for future reanalysis")

# Uncomment and run the evaluation below
# results_extended = analyzer.evaluate_our_method_batched(
#     num_candidates=num_candidates,
#     num_trials=num_trials,
#     aggregation_method="mean",
#     batch_size=batch_size,
#     use_parallel=True,
#     precalculated_bias=prebias_gpt35_books_100,
#     checkpoint_file="evaluation_checkpoint_bias20_books_extended.json"
# )

print("\n⚠️  Uncomment the evaluation code above to run the extended experiment")
print("✅ Extended setup complete - ready to run evaluation!")


In [ ]:
# Compute bias analysis with different list size (100 list size)
print("🔍 Computing bias analysis with 100 list size...")

# Uncomment the line below to run bias analysis with larger list size
# bias_analysis_list100 = analyzer.compute_bias_analysis(5, None, True, None, 100)
# print(bias_analysis_list100)

# Placeholder values for 100 list size bias analysis - replace with actual results
prebias_gpt35_books_list100 = {
    'avg_primacy': 0.145,  # Placeholder - replace with actual computed values
    'avg_recency': 0.155,  # Placeholder - replace with actual computed values
    'avg_middle': 1.700    # Placeholder - replace with actual computed values
}

print("📊 Bias scores with 100 list size (placeholder):")
print(f"  Primacy: {prebias_gpt35_books_list100['avg_primacy']:.3f}")
print(f"  Recency: {prebias_gpt35_books_list100['avg_recency']:.3f}")
print(f"  Middle: {prebias_gpt35_books_list100['avg_middle']:.3f}")


In [ ]:
# Results Analysis and Comparison
print("📊 RESULTS ANALYSIS")
print("=" * 50)

# This cell will contain analysis of results once experiments are run
print("🔍 Analysis will be available after running experiments above")
print("\nTo analyze results:")
print("1. 📈 Run bias detection experiments")
print("2. 🚀 Run evaluation experiments")
print("3. 📊 Compare results with benchmarks")
print("4. 💾 Save results to appropriate files")

print("\n📋 Expected output metrics:")
print("- Accuracy")
print("- NDCG@1, NDCG@5, NDCG@10, NDCG@20")
print("- Comparison with Raw Output, Bootstrapping, and STELLA methods")
print("- Statistical significance analysis")

print("\n⚠️  Run experiments above to populate this analysis section")
